<a href="https://colab.research.google.com/github/Zaneframe/CSCC102-FileHandling/blob/main/Lab9_File_Handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CVS File

In [ ]:
import io
import pandas as pd
from google.colab import files

# Ask the user for their details
student_name = input("Enter your name: ")
course_name = input("Enter your Course & Major (e.g., BSCS): ")
year_level = input("Enter your Year Level (e.g., 1st Year): ")

# Display the personalized header and grade report
print("\n" + "="*40)
print(f"GRADE REPORT FOR: {student_name.strip().upper()}")
print(f"COURSE & MAJOR:   {course_name.strip().upper()}")
print(f"YEAR LEVEL:       {year_level.strip().upper()}")
print("="*40 + "\n")

# 1. Trigger the file upload prompt
uploaded = files.upload()

# Helper function to find the header row index and read df correctly
def read_grades_df(file_bytes, is_excel=False):
    # Read raw data first to inspect structure
    if is_excel:
        raw_df = pd.read_excel(io.BytesIO(file_bytes), header=None)
    else:
        raw_df = pd.read_csv(io.BytesIO(file_bytes), header=None)

    # Find row index containing typical headers
    header_row_idx = 0
    for i in range(len(raw_df)):
        row_vals = [str(val).strip().lower() for val in raw_df.iloc[i].values if pd.notna(val)]
        if 'units' in row_vals or 'subject' in row_vals or 'grade' in row_vals:
            header_row_idx = i
            break

    # Re-read utilizing the correct header row offset
    if is_excel:
        return pd.read_excel(io.BytesIO(file_bytes), skiprows=header_row_idx)
    else:
        return pd.read_csv(io.BytesIO(file_bytes), skiprows=header_row_idx)

# 2. Dynamically get the name of the uploaded file
if uploaded:
    file_name = list(uploaded.keys())[0]

    # Check file extension to read it correctly
    is_excel = file_name.endswith('.xlsx') or file_name.endswith('.xls')
    df = read_grades_df(uploaded[file_name], is_excel=is_excel)

    # Normalize column names by stripping whitespaces
    df.columns = [str(col).strip() for col in df.columns]

    # 3. View the full data
    display(df)
else:
    print('No file was uploaded.')

# Calculate and print grade report statistics
if 'df' in globals() and df is not None:
    # Ensure we target the right columns case-insensitively if needed
    units_col = next((col for col in df.columns if col.lower() == 'units'), 'Units')
    grade_col = next((col for col in df.columns if col.lower() == 'grade'), 'Grade')

    # 1. Total Number of Subjects
    num_subjects = len(df)

    # 2. Total Units (ensure they are numeric)
    df[units_col] = pd.to_numeric(df[units_col], errors='coerce')
    total_units = df[units_col].sum()

    # 3. GWA (Grade Weighted Average)
    # Ensure Grade is numeric
    df[grade_col] = pd.to_numeric(df[grade_col], errors='coerce')

    # Calculate GWA: sum(Grade * Units) / sum(Units)
    weighted_sum = (df[grade_col] * df[units_col]).sum()
    gwa = weighted_sum / total_units if total_units > 0 else 0

    print("="*40)
    print("GRADE SUMMARY")
    print("="*40)
    print(f"Student Name:                   {student_name.strip().upper()}")
    print(f"Course & Major:                 {course_name.strip().upper()}")
    print(f"Year Level:                     {year_level.strip().upper()}")
    print(f"Number of Subjects:             {num_subjects}")
    print(f"Total Units:                    {total_units:.1f}")
    print(f"GWA Grade Weighted Average:     {gwa:.2f}")
    print("="*40)
else:
    print("DataFrame 'df' is not loaded. Please upload the file in the previous step first.")

Enter your name: Mike Shi
Enter your Course & Major (e.g., BSCS): BSCS
Enter your Year Level (e.g., 1st Year): 1st

GRADE REPORT FOR: MIKE SHI
COURSE & MAJOR:   BSCS
YEAR LEVEL:       1ST



Saving Shi Grade Report.csv to Shi Grade Report (2).csv


,Subject,Description,Units,Teacher,Grade
0,CSCC101,Introduction To Computing,3,Ms. Arianne,1.50
1,CSCC102,Fundamentals of Programming,3,Ms. Brenda,1.75
2,CSMath,Linear Algebra,3,Sir Victorino,1.50
3,MST111,Mathematic in Modern World,3,Ms. Nathalie,1.50
4,RE111,Salvation History: Old Testament,3,Ms. Vega,1.75
5,SDS-1,Student Development Session,1,Ms. Gevero,1.00
6,PE-1,Pathfit 1 Physical Activities Towards Health a...,2,Sir Dave,1.25
7,SOCSCI-143,Rizal's Life Work And Writing,3,Ms. Marcela,2.00
8,WKL 1,"Introduksiyon sa Filipinolohiya ( Wika,Kultura...",3,Ms. Umadhay,1.50
9,ROTC,Reserve Officers' Training Corps ( 1st Platoon...,3,Sir Tenorio,1.00


GRADE SUMMARY
Student Name:                   MIKE SHI
Course & Major:                 BSCS
Year Level:                     1ST
Number of Subjects:             10
Total Units:                    27.0
GWA Grade Weighted Average:     1.52


# Display

In [ ]:
import io
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# 1. Ask the user for their academic details
student_name = input("Enter your name: ")
course_name = input("Enter your Course & Major (e.g., BSCS): ")
year_level = input("Enter your Year Level (e.g., 1st Year): ")

# 2. Trigger the file upload prompt
print("\nPlease upload your grade sheet below:")
uploaded = files.upload()

# CSS styling with full print color preservation
report_style = """
<style>
    /* CSS Print Settings to retain green theme in PDF print output */
    @media print {
        body {
            background-color: #ffffff !important;
            -webkit-print-color-adjust: exact !important;
            print-color-adjust: exact !important;
        }
        body * {
            visibility: hidden;
        }
        .report-card, .report-card * {
            visibility: visible;
        }
        .report-card {
            position: absolute;
            left: 50%;
            top: 20px;
            transform: translateX(-50%);
            width: 100%;
            max-width: 650px;
            border: 2px solid #005F30 !important;
            border-radius: 14px !important;
            background-color: #ffffff !important;
            margin: 0 !important;
            padding: 28px 32px !important;
            box-shadow: none !important;
            -webkit-print-color-adjust: exact !important;
            print-color-adjust: exact !important;
        }
        .no-print {
            display: none !important;
        }
    }

    .report-card {
        background-color: #ffffff;
        border: 2px solid #005F30;
        border-radius: 14px;
        padding: 28px 32px;
        max-width: 650px;
        margin: 15px auto;
        font-family: 'Segoe UI', Arial, sans-serif;
        box-shadow: 0px 6px 20px rgba(0, 95, 48, 0.12);
        color: #1e1b4b;
        -webkit-print-color-adjust: exact;
        print-color-adjust: exact;
    }
    .report-header {
        text-align: center;
        font-size: 20px;
        font-weight: 800;
        color: #005F30;
        text-transform: uppercase;
        letter-spacing: 1.2px;
        margin-bottom: 22px;
        border-bottom: 2px solid #005F30;
        padding-bottom: 10px;
        -webkit-print-color-adjust: exact;
        print-color-adjust: exact;
    }
    .student-details {
        font-size: 13.5px;
        line-height: 1.8;
        margin-bottom: 20px;
    }
    .student-details table {
        border-collapse: collapse;
        width: 100%;
    }
    .student-details td {
        padding: 2px 10px 2px 0;
        vertical-align: top;
    }
    .student-details td.label {
        font-weight: 700;
        color: #005F30;
        width: 140px;
    }
    .section-title {
        font-size: 12.5px;
        font-weight: 800;
        color: #005F30;
        text-transform: uppercase;
        margin-top: 20px;
        margin-bottom: 10px;
        letter-spacing: 0.5px;
    }
    .grades-table {
        width: 100%;
        border-collapse: collapse;
        font-size: 12.5px;
        margin-bottom: 20px;
        background-color: #ffffff;
        border-radius: 6px;
        overflow: hidden;
    }
    .grades-table th {
        background-color: #e6f4ea !important;
        color: #005F30 !important;
        font-weight: 700;
        text-align: left;
        padding: 8px 10px;
        border-bottom: 2px solid #005F30;
        -webkit-print-color-adjust: exact;
        print-color-adjust: exact;
    }
    .grades-table td {
        padding: 7px 10px;
        border-bottom: 1px solid #e6f4ea;
        color: #334155;
    }
    .grades-table tr:nth-child(even) {
        background-color: #f9fdfa !important;
        -webkit-print-color-adjust: exact;
        print-color-adjust: exact;
    }
    .summary-table {
        width: 100%;
        border-collapse: collapse;
        font-size: 13px;
        margin-bottom: 10px;
    }
    .summary-table td {
        padding: 6px 0;
        color: #334155;
    }
    .summary-table td.val {
        text-align: right;
        font-weight: 700;
        color: #005F30;
    }
    .gwa-banner {
        background-color: #005F30 !important;
        color: #ffffff !important;
        padding: 12px 18px;
        border-radius: 8px;
        display: flex;
        justify-content: space-between;
        align-items: center;
        font-weight: bold;
        margin-top: 15px;
        -webkit-print-color-adjust: exact;
        print-color-adjust: exact;
    }
    .gwa-title {
        font-size: 14px;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }
    .gwa-value {
        font-size: 20px;
    }
    .print-btn {
        display: block;
        width: 100%;
        background-color: #005F30;
        color: white;
        border: none;
        padding: 10px;
        font-size: 14px;
        font-weight: bold;
        text-transform: uppercase;
        border-radius: 6px;
        cursor: pointer;
        margin-top: 20px;
        text-align: center;
        transition: background-color 0.2s;
    }
    .print-btn:hover {
        background-color: #004d27;
    }
</style>
"""

# Helper function to find the header row index and read df correctly
def read_grades_df(file_bytes, is_excel=False):
    # Read raw data first to inspect structure
    if is_excel:
        raw_df = pd.read_excel(io.BytesIO(file_bytes), header=None)
    else:
        raw_df = pd.read_csv(io.BytesIO(file_bytes), header=None)

    # Find row index containing typical headers
    header_row_idx = 0
    for i in range(len(raw_df)):
        row_vals = [str(val).strip().lower() for val in raw_df.iloc[i].values if pd.notna(val)]
        if 'units' in row_vals or 'subject' in row_vals or 'grade' in row_vals:
            header_row_idx = i
            break

    # Re-read utilizing the correct header row offset
    if is_excel:
        return pd.read_excel(io.BytesIO(file_bytes), skiprows=header_row_idx)
    else:
        return pd.read_csv(io.BytesIO(file_bytes), skiprows=header_row_idx)

# 3. Read spreadsheet data and display output
if uploaded:
    file_name = list(uploaded.keys())[0]
    is_excel = file_name.endswith('.xlsx') or file_name.endswith('.xls')
    df = read_grades_df(uploaded[file_name], is_excel=is_excel)

    # Normalize column names by stripping whitespaces
    df.columns = [str(col).strip() for col in df.columns]

    # Identify columns dynamically using lowercase fallbacks
    units_col = next((col for col in df.columns if col.lower() == 'units'), 'Units')
    grade_col = next((col for col in df.columns if col.lower() == 'grade'), 'Grade')

    # Total Number of Subjects
    num_subjects = len(df)

    # Ensure Units and Grade are numeric
    df[units_col] = pd.to_numeric(df[units_col], errors='coerce')
    df[grade_col] = pd.to_numeric(df[grade_col], errors='coerce')
    total_units = df[units_col].sum()

    # Calculate weighted sum and GWA
    weighted_sum = (df[grade_col] * df[units_col]).sum()
    calculated_gwa = weighted_sum / total_units if total_units > 0 else 0.0

    # Dynamic table generation for all uploaded columns
    table_headers_html = "".join([f"<th>{col}</th>" for col in df.columns])

    table_rows_html = ""
    for idx, row in df.iterrows():
        table_rows_html += "<tr>"
        for col in df.columns:
            val = row[col]
            if col.lower() == 'grade' and isinstance(val, (int, float)) and not pd.isna(val):
                formatted_val = f"{val:.2f}"
            elif col.lower() == 'units' and isinstance(val, (int, float)) and not pd.isna(val):
                formatted_val = f"{val:.1f}" if val % 1 != 0 else f"{int(val)}"
            else:
                formatted_val = str(val) if pd.notna(val) else ""
            table_rows_html += f"<td>{formatted_val}</td>"
        table_rows_html += "</tr>"

    s_name = student_name.strip().upper()
    c_name = course_name.strip().upper()
    y_level = year_level.strip().upper()

    report_card_html = f"""
    {report_style}
    <div class="report-card">
        <div class="report-header">Academic Grade Report</div>

        <div class="student-details">
            <table>
                <tr>
                    <td class="label">STUDENT NAME:</td>
                    <td>{s_name}</td>
                </tr>
                <tr>
                    <td class="label">COURSE & MAJOR:</td>
                    <td>{c_name}</td>
                </tr>
                <tr>
                    <td class="label">YEAR LEVEL:</td>
                    <td>{y_level}</td>
                </tr>
            </table>
        </div>

        <div class="section-title">Course Breakdown</div>
        <table class="grades-table">
            <thead>
                <tr>{table_headers_html}</tr>
            </thead>
            <tbody>
                {table_rows_html}
            </tbody>
        </table>

        <div class="section-title">Summary Metrics</div>
        <table class="summary-table">
            <tr>
                <td>Total Evaluated Subjects</td>
                <td class="val">{num_subjects}</td>
            </tr>
            <tr>
                <td>Total Credit Units</td>
                <td class="val">{total_units:.1f}</td>
            </tr>
        </table>

        <div class="gwa-banner">
            <span class="gwa-title">General Weighted Average (GWA)</span>
            <span class="gwa-value">{calculated_gwa:.2f}</span>
        </div>

        <button class="print-btn no-print" onclick="window.print()">Print to PDF</button>
    </div>
    """
    display(HTML(report_card_html))
else:
    print("No file was uploaded.")

Enter your name: Mike Shi
Enter your Course & Major (e.g., BSCS): BSCS
Enter your Year Level (e.g., 1st Year): 1st

Please upload your grade sheet below:


Saving Shi Grade Report.csv to Shi Grade Report (4).csv


# Teacher Files

In [ ]:
import io
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

print("Please upload the combined multi-student spreadsheet:")
uploaded_combined = files.upload()

if uploaded_combined:
    file_name = list(uploaded_combined.keys())[0]
    # Read raw Excel data with no header initially to scan for student blocks
    if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
        raw_df = pd.read_excel(io.BytesIO(uploaded_combined[file_name]), header=None)
    else:
        raw_df = pd.read_csv(io.BytesIO(uploaded_combined[file_name]), header=None)

    students_data = {}
    current_student = None
    current_program = ""
    current_rows = []
    table_headers = ['Code', 'Description', 'Units', 'Teacher', 'Grade']

    # Walk through the Excel rows
    i = 0
    while i < len(raw_df):
        row = raw_df.iloc[i]
        row_vals = [val for val in row.values if pd.notna(val)]
        row_vals_str = [str(val).strip() for val in row_vals]

        # Skip empty rows
        if not row_vals_str:
            i += 1
            continue

        row_lower = [v.lower() for v in row_vals_str]

        # If we see the table header row, the previous row must have been the student info
        if 'code' in row_lower and 'description' in row_lower and 'grade' in row_lower:
            # Grab the actual table headers
            table_headers = [str(v).strip() for v in row.values if pd.notna(v)]

            # Let's inspect the row above this header row to identify the student name & program
            prev_row_idx = i - 1
            while prev_row_idx >= 0:
                prev_row = raw_df.iloc[prev_row_idx]
                prev_vals = [str(val).strip() for val in prev_row.values if pd.notna(val)]
                if prev_vals:
                    # Save the previous student details before switching to a new student
                    if current_student and current_rows:
                        students_data[current_student] = {
                            'program': current_program,
                            'headers': table_headers,
                            'rows': current_rows.copy()
                        }
                        current_rows = []

                    current_student = prev_vals[0]
                    current_program = prev_vals[1] if len(prev_vals) > 1 else ""
                    break
                prev_row_idx -= 1
            i += 1
            continue

        # If we have an active student and the row looks like standard grade data (Code, Title, Units, Teacher, Grade)
        if current_student is not None:
            # Keep appending rows that look like subject codes (e.g. length of values >= 3)
            # We filter out rows that might contain metadata, empty values, or text from headers
            if len(row_vals_str) >= 3 and not any(x in row_lower for x in ['code', 'description', 'gwa']):
                current_rows.append([val if pd.notna(val) else "" for val in row.values[:len(table_headers)]])

        i += 1

    # Save the last student in the file
    if current_student and current_rows:
        students_data[current_student] = {
            'program': current_program,
            'headers': table_headers,
            'rows': current_rows.copy()
        }

    # Create dropdown with the discovered students
    student_selector = widgets.Dropdown(
        options=list(students_data.keys()),
        description='Select Student:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    output_area = widgets.Output()

    def show_student_report(change):
        with output_area:
            clear_output()
            student = change['new']
            if not student:
                return

            data = students_data[student]
            headers = data['headers']
            rows = data['rows']

            # Convert list of rows to DataFrame
            student_df = pd.DataFrame(rows, columns=headers[:len(rows[0])])

            # Identify Units and Grade columns
            units_col = next((col for col in student_df.columns if col.lower() == 'units'), None)
            grade_col = next((col for col in student_df.columns if col.lower() == 'grade'), None)

            if units_col and grade_col:
                student_df[units_col] = pd.to_numeric(student_df[units_col], errors='coerce')
                student_df[grade_col] = pd.to_numeric(student_df[grade_col], errors='coerce')
                valid_df = student_df.dropna(subset=[units_col, grade_col])

                total_units = valid_df[units_col].sum()
                weighted_sum = (valid_df[grade_col] * valid_df[units_col]).sum()
                gwa = weighted_sum / total_units if total_units > 0 else 0.0
                num_subjects = len(valid_df)
            else:
                total_units = 0
                gwa = 0.0
                num_subjects = len(student_df)

            # Generate HTML table cells
            table_headers_html = "".join([f"<th>{col}</th>" for col in headers])
            table_rows_html = ""
            for _, r in student_df.iterrows():
                table_rows_html += "<tr>"
                for col in student_df.columns:
                    val = r[col]
                    if col.lower() == 'grade' and isinstance(val, (int, float)) and not pd.isna(val):
                        formatted_val = f"{val:.2f}"
                    elif col.lower() == 'units' and isinstance(val, (int, float)) and not pd.isna(val):
                        formatted_val = f"{val:.1f}" if val % 1 != 0 else f" {int(val)}"
                    else:
                        formatted_val = str(val) if pd.notna(val) else ""
                    table_rows_html += f"<td>{formatted_val}</td>"
                table_rows_html += "</tr>"

            # Style definitions for Grade Report Card with Print Compatibility
            report_style = """
            <style>
                @media print {
                    body {
                        background-color: #ffffff !important;
                        -webkit-print-color-adjust: exact !important;
                        print-color-adjust: exact !important;
                    }
                    body * {
                        visibility: hidden;
                    }
                    .teacher-report-card, .teacher-report-card * {
                        visibility: visible;
                    }
                    .teacher-report-card {
                        position: absolute;
                        left: 50%;
                        top: 20px;
                        transform: translateX(-50%);
                        width: 100%;
                        max-width: 650px;
                        border: 2px solid #005F30 !important;
                        border-radius: 14px !important;
                        background-color: #ffffff !important;
                        margin: 0 !important;
                        padding: 28px 32px !important;
                        box-shadow: none !important;
                        -webkit-print-color-adjust: exact !important;
                        print-color-adjust: exact !important;
                    }
                    .no-print {
                        display: none !important;
                    }
                }

                .teacher-report-card {
                    background-color: #ffffff;
                    border: 2px solid #005F30;
                    border-radius: 14px;
                    padding: 28px 32px;
                    max-width: 650px;
                    margin: 15px auto;
                    font-family: 'Segoe UI', Arial, sans-serif;
                    box-shadow: 0px 6px 20px rgba(0, 95, 48, 0.12);
                    color: #1e1b4b;
                    -webkit-print-color-adjust: exact;
                    print-color-adjust: exact;
                }
                .report-header {
                    text-align: center;
                    font-size: 20px;
                    font-weight: 800;
                    color: #005F30;
                    text-transform: uppercase;
                    letter-spacing: 1.2px;
                    margin-bottom: 22px;
                    border-bottom: 2px solid #005F30;
                    padding-bottom: 10px;
                    -webkit-print-color-adjust: exact;
                    print-color-adjust: exact;
                }
                .student-details {
                    font-size: 13.5px;
                    line-height: 1.8;
                    margin-bottom: 20px;
                }
                .student-details table {
                    border-collapse: collapse;
                    width: 100%;
                }
                .student-details td.label {
                    font-weight: 700;
                    color: #005F30;
                    width: 140px;
                }
                .section-title {
                    font-size: 12.5px;
                    font-weight: 800;
                    color: #005F30;
                    text-transform: uppercase;
                    margin-top: 20px;
                    margin-bottom: 10px;
                    letter-spacing: 0.5px;
                }
                .grades-table {
                    width: 100%;
                    border-collapse: collapse;
                    font-size: 12.5px;
                    margin-bottom: 20px;
                }
                .grades-table th {
                    background-color: #e6f4ea !important;
                    color: #005F30 !important;
                    font-weight: 700;
                    text-align: left;
                    padding: 8px 10px;
                    border-bottom: 2px solid #005F30;
                    -webkit-print-color-adjust: exact;
                    print-color-adjust: exact;
                }
                .grades-table td {
                    padding: 7px 10px;
                    border-bottom: 1px solid #e6f4ea;
                    color: #334155;
                }
                .summary-table {
                    width: 100%;
                    border-collapse: collapse;
                    font-size: 13px;
                }
                .summary-table td {
                    padding: 6px 0;
                }
                .summary-table td.val {
                    text-align: right;
                    font-weight: 700;
                    color: #005F30;
                }
                .gwa-banner {
                    background-color: #005F30 !important;
                    color: #ffffff !important;
                    padding: 12px 18px;
                    border-radius: 8px;
                    display: flex;
                    justify-content: space-between;
                    align-items: center;
                    font-weight: bold;
                    margin-top: 15px;
                    -webkit-print-color-adjust: exact;
                    print-color-adjust: exact;
                }
                .gwa-value {
                    font-size: 20px;
                }
                .print-btn {
                    display: block;
                    width: 100%;
                    background-color: #005F30;
                    color: white;
                    border: none;
                    padding: 10px;
                    font-size: 14px;
                    font-weight: bold;
                    text-transform: uppercase;
                    border-radius: 6px;
                    cursor: pointer;
                    margin-top: 20px;
                    text-align: center;
                    transition: background-color 0.2s;
                }
                .print-btn:hover {
                    background-color: #004d27;
                }
            </style>
            """.strip()

            report_card_html = f"""
            {report_style}
            <div class="teacher-report-card">
                <div class="report-header">Academic Grade Report</div>

                <div class="student-details">
                    <table>
                        <tr>
                            <td class="label">STUDENT NAME:</td>
                            <td>{student.upper()}</td>
                        </tr>
                        <tr>
                            <td class="label">COURSE & PROGRAM:</td>
                            <td>{data["program"].upper()}</td>
                        </tr>
                    </table>
                </div>

                <div class="section-title">Course Breakdown</div>
                <table class="grades-table">
                    <thead>
                        <tr>{table_headers_html}</tr>
                    </thead>
                    <tbody>
                        {table_rows_html}
                    </tbody>
                </table>

                <div class="section-title">Summary Metrics</div>
                <table class="summary-table">
                    <tr>
                        <td>Total Evaluated Subjects</td>
                        <td class="val">{num_subjects}</td>
                    </tr>
                    <tr>
                        <td>Total Credit Units</td>
                        <td class="val">{total_units:.1f}</td>
                    </tr>
                </table>

                <div class="gwa-banner">
                    <span>General Weighted Average (GWA)</span>
                    <span class="gwa-value">{gwa:.2f}</span>
                </div>

                <button class="print-btn no-print" onclick="window.print()">Print to PDF</button>
            </div>
            """
            display(HTML(report_card_html))

    student_selector.observe(show_student_report, names='value')
    display(student_selector)
    display(output_area)

    # Trigger default option display
    if list(students_data.keys()):
        student_selector.value = list(students_data.keys())[0]
else:
    print("No file was selected.")

Please upload the combined multi-student spreadsheet:


Saving 11 Studetns Report.csv to 11 Studetns Report.csv


Dropdown(description='Select Student:', layout=Layout(width='400px'), options=('Mike Jeffrey O. Shi', 'Alexand…

Output()

# Ranking System

In [1]:
import io
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# ── GWA Remark Scale (Philippine 1.0–5.0 system) ──────────────────────────────
def get_gwa_remark(gwa):
    if gwa <= 0:      return ("N/A",            "#6b7280")
    elif gwa <= 1.50: return ("Excellent",       "#15803d")
    elif gwa <= 1.75: return ("Very Good",       "#16a34a")
    elif gwa <= 2.00: return ("Good",            "#65a30d")
    elif gwa <= 2.50: return ("Satisfactory",    "#ca8a04")
    elif gwa <= 3.00: return ("Passing",         "#d97706")
    else:             return ("Failed",          "#dc2626")

print("Please upload the combined multi-student spreadsheet:")
uploaded_combined = files.upload()

if uploaded_combined:
    file_name = list(uploaded_combined.keys())[0]
    if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
        raw_df = pd.read_excel(io.BytesIO(uploaded_combined[file_name]), header=None)
    else:
        raw_df = pd.read_csv(io.BytesIO(uploaded_combined[file_name]), header=None)

    students_data = {}
    current_student = None
    current_program = ""
    current_rows = []
    table_headers = ['Code', 'Description', 'Units', 'Teacher', 'Grade']

    i = 0
    while i < len(raw_df):
        row = raw_df.iloc[i]
        row_vals = [val for val in row.values if pd.notna(val)]
        row_vals_str = [str(val).strip() for val in row_vals]

        if not row_vals_str:
            i += 1
            continue

        row_lower = [v.lower() for v in row_vals_str]

        if 'code' in row_lower and 'description' in row_lower and 'grade' in row_lower:
            table_headers = [str(v).strip() for v in row.values if pd.notna(v)]
            prev_row_idx = i - 1
            while prev_row_idx >= 0:
                prev_row = raw_df.iloc[prev_row_idx]
                prev_vals = [str(val).strip() for val in prev_row.values if pd.notna(val)]
                if prev_vals:
                    if current_student and current_rows:
                        students_data[current_student] = {
                            'program': current_program,
                            'headers': table_headers,
                            'rows': current_rows.copy()
                        }
                        current_rows = []
                    current_student = prev_vals[0]
                    current_program = prev_vals[1] if len(prev_vals) > 1 else ""
                    break
                prev_row_idx -= 1
            i += 1
            continue

        if current_student is not None:
            if len(row_vals_str) >= 3 and not any(x in row_lower for x in ['code', 'description', 'gwa']):
                current_rows.append([val if pd.notna(val) else "" for val in row.values[:len(table_headers)]])

        i += 1

    if current_student and current_rows:
        students_data[current_student] = {
            'program': current_program,
            'headers': table_headers,
            'rows': current_rows.copy()
        }

    # ── Pre-compute GWA for all students ──────────────────────────────────────
    def compute_gwa(data):
        rows = data['rows']
        if not rows:
            return 0.0
        df = pd.DataFrame(rows, columns=data['headers'][:len(rows[0])])
        u = next((c for c in df.columns if c.lower() == 'units'), None)
        g = next((c for c in df.columns if c.lower() == 'grade'), None)
        if u and g:
            df[u] = pd.to_numeric(df[u], errors='coerce')
            df[g] = pd.to_numeric(df[g], errors='coerce')
            v = df.dropna(subset=[u, g])
            total = v[u].sum()
            return (v[g] * v[u]).sum() / total if total > 0 else 0.0
        return 0.0

    student_gwas = {s: compute_gwa(d) for s, d in students_data.items()}
    # Lower GWA = better in PH system; push 0.0 (no data) to end
    ranked = sorted(student_gwas.items(), key=lambda x: (x[1] == 0, x[1]))


    def build_scoreboard():
        medal = {1: "🥇", 2: "🥈", 3: "🥉"}
        rows_html = ""
        for rank, (student, gwa) in enumerate(ranked, 1):
            remark, color = get_gwa_remark(gwa)
            icon = medal.get(rank, f"#{rank}")
            program = students_data[student]['program']
            gwa_display = f"{gwa:.2f}" if gwa > 0 else "—"
            rows_html += f"""<tr>
                <td style="text-align:center;font-weight:700;font-size:15px">{icon}</td>
                <td style="font-weight:600">{student.upper()}</td>
                <td style="font-size:12px;color:#6b7280">{program}</td>
                <td style="text-align:center;font-weight:700;color:#005F30">{gwa_display}</td>
                <td style="text-align:center">
                    <span style="background:{color};color:#fff;padding:3px 10px;
                                 border-radius:12px;font-size:12px;font-weight:600">
                        {remark}
                    </span>
                </td>
            </tr>"""

        return f"""
        <style>
            .sb{{background:#fff;border:2px solid #005F30;border-radius:14px;padding:24px 28px;
                 max-width:700px;margin:15px auto;font-family:'Segoe UI',Arial,sans-serif;
                 box-shadow:0 6px 20px rgba(0,95,48,.12)}}
            .sb-hdr{{text-align:center;font-size:20px;font-weight:800;color:#005F30;
                     text-transform:uppercase;letter-spacing:1.2px;
                     border-bottom:2px solid #005F30;padding-bottom:10px;margin-bottom:18px}}
            .sb-legend{{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:14px;font-size:12px}}
            .sb-legend span{{padding:3px 10px;border-radius:12px;color:#fff;font-weight:600}}
            .sb-tbl{{width:100%;border-collapse:collapse;font-size:13px}}
            .sb-tbl th{{background:#e6f4ea;color:#005F30;font-weight:700;
                        padding:8px 10px;text-align:left;border-bottom:2px solid #005F30}}
            .sb-tbl td{{padding:8px 10px;border-bottom:1px solid #e6f4ea;color:#334155}}
            .sb-tbl tr:hover td{{background:#f0fdf4}}
        </style>
        <div class="sb">
            <div class="sb-hdr">🏆 Student Rankings — GWA Scoreboard</div>
            <div class="sb-legend">
                <span style="background:#15803d">1.00–1.50 Excellent</span>
                <span style="background:#16a34a">1.51–1.75 Very Good</span>
                <span style="background:#65a30d">1.76–2.00 Good</span>
                <span style="background:#ca8a04">2.01–2.50 Satisfactory</span>
                <span style="background:#d97706">2.51–3.00 Passing</span>
                <span style="background:#dc2626">3.01+ Failed</span>
            </div>
            <table class="sb-tbl">
                <thead><tr>
                    <th style="width:55px">Rank</th>
                    <th>Student Name</th>
                    <th>Program</th>
                    <th style="text-align:center">GWA</th>
                    <th style="text-align:center">Remarks</th>
                </tr></thead>
                <tbody>{rows_html}</tbody>
            </table>
        </div>"""

    student_selector = widgets.Dropdown(
        options=list(students_data.keys()),
        description='Select Student:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    report_output   = widgets.Output()
    scoreboard_output = widgets.Output()

    with scoreboard_output:
        display(HTML(build_scoreboard()))

    def show_student_report(change):
        with report_output:
            clear_output()
            student = change['new']
            if not student:
                return

            data    = students_data[student]
            headers = data['headers']
            rows    = data['rows']

            student_df = pd.DataFrame(rows, columns=headers[:len(rows[0])])
            units_col  = next((c for c in student_df.columns if c.lower() == 'units'), None)
            grade_col  = next((c for c in student_df.columns if c.lower() == 'grade'), None)

            if units_col and grade_col:
                student_df[units_col] = pd.to_numeric(student_df[units_col], errors='coerce')
                student_df[grade_col] = pd.to_numeric(student_df[grade_col], errors='coerce')
                valid_df    = student_df.dropna(subset=[units_col, grade_col])
                total_units = valid_df[units_col].sum()
                weighted_sum = (valid_df[grade_col] * valid_df[units_col]).sum()
                gwa         = weighted_sum / total_units if total_units > 0 else 0.0
                num_subjects = len(valid_df)
            else:
                total_units  = 0
                gwa          = 0.0
                num_subjects = len(student_df)

            remark, remark_color = get_gwa_remark(gwa)
            rank = next((i + 1 for i, (s, _) in enumerate(ranked) if s == student), "N/A")

            table_headers_html = "".join([f"<th>{col}</th>" for col in headers])
            table_rows_html    = ""
            for _, r in student_df.iterrows():
                table_rows_html += "<tr>"
                for col in student_df.columns:
                    val = r[col]
                    if col.lower() == 'grade' and isinstance(val, (int, float)) and not pd.isna(val):
                        formatted_val = f"{val:.2f}"
                    elif col.lower() == 'units' and isinstance(val, (int, float)) and not pd.isna(val):
                        formatted_val = f"{val:.1f}" if val % 1 != 0 else f" {int(val)}"
                    else:
                        formatted_val = str(val) if pd.notna(val) else ""
                    table_rows_html += f"<td>{formatted_val}</td>"
                table_rows_html += "</tr>"

            report_style = """
            <style>
                @media print {
                    body { background-color: #ffffff !important; -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }
                    body * { visibility: hidden; }
                    .teacher-report-card, .teacher-report-card * { visibility: visible; }
                    .teacher-report-card {
                        position: absolute; left: 50%; top: 20px; transform: translateX(-50%);
                        width: 100%; max-width: 650px; border: 2px solid #005F30 !important;
                        border-radius: 14px !important; background-color: #ffffff !important;
                        margin: 0 !important; padding: 28px 32px !important; box-shadow: none !important;
                        -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important;
                    }
                    .no-print { display: none !important; }
                }
                .teacher-report-card {
                    background-color: #ffffff; border: 2px solid #005F30; border-radius: 14px;
                    padding: 28px 32px; max-width: 650px; margin: 15px auto;
                    font-family: 'Segoe UI', Arial, sans-serif;
                    box-shadow: 0px 6px 20px rgba(0, 95, 48, 0.12); color: #1e1b4b;
                    -webkit-print-color-adjust: exact; print-color-adjust: exact;
                }
                .report-header {
                    text-align: center; font-size: 20px; font-weight: 800; color: #005F30;
                    text-transform: uppercase; letter-spacing: 1.2px; margin-bottom: 22px;
                    border-bottom: 2px solid #005F30; padding-bottom: 10px;
                    -webkit-print-color-adjust: exact; print-color-adjust: exact;
                }
                .student-details { font-size: 13.5px; line-height: 1.8; margin-bottom: 20px; }
                .student-details table { border-collapse: collapse; width: 100%; }
                .student-details td.label { font-weight: 700; color: #005F30; width: 140px; }
                .section-title {
                    font-size: 12.5px; font-weight: 800; color: #005F30;
                    text-transform: uppercase; margin-top: 20px; margin-bottom: 10px; letter-spacing: 0.5px;
                }
                .grades-table { width: 100%; border-collapse: collapse; font-size: 12.5px; margin-bottom: 20px; }
                .grades-table th {
                    background-color: #e6f4ea !important; color: #005F30 !important;
                    font-weight: 700; text-align: left; padding: 8px 10px;
                    border-bottom: 2px solid #005F30;
                    -webkit-print-color-adjust: exact; print-color-adjust: exact;
                }
                .grades-table td { padding: 7px 10px; border-bottom: 1px solid #e6f4ea; color: #334155; }
                .summary-table { width: 100%; border-collapse: collapse; font-size: 13px; }
                .summary-table td { padding: 6px 0; }
                .summary-table td.val { text-align: right; font-weight: 700; color: #005F30; }
                .gwa-banner {
                    background-color: #005F30 !important; color: #ffffff !important;
                    padding: 12px 18px; border-radius: 8px;
                    display: flex; justify-content: space-between; align-items: center;
                    font-weight: bold; margin-top: 15px;
                    -webkit-print-color-adjust: exact; print-color-adjust: exact;
                }
                .gwa-value { font-size: 20px; }
                .rank-bar {
                    display: flex; justify-content: space-between; align-items: center;
                    margin-top: 10px; padding: 8px 14px;
                    background: #f0fdf4; border-radius: 8px; font-size: 13px; color: #374151;
                }
                .print-btn {
                    display: block; width: 100%; background-color: #005F30; color: white; border: none;
                    padding: 10px; font-size: 14px; font-weight: bold; text-transform: uppercase;
                    border-radius: 6px; cursor: pointer; margin-top: 20px; text-align: center;
                }
                .print-btn:hover { background-color: #004d27; }
            </style>
            """

            report_card_html = f"""
            {report_style}
            <div class="teacher-report-card">
                <div class="report-header">Academic Grade Report</div>

                <div class="student-details">
                    <table>
                        <tr><td class="label">STUDENT NAME:</td><td>{student.upper()}</td></tr>
                        <tr><td class="label">COURSE &amp; PROGRAM:</td><td>{data["program"].upper()}</td></tr>
                    </table>
                </div>

                <div class="section-title">Course Breakdown</div>
                <table class="grades-table">
                    <thead><tr>{table_headers_html}</tr></thead>
                    <tbody>{table_rows_html}</tbody>
                </table>

                <div class="section-title">Summary Metrics</div>
                <table class="summary-table">
                    <tr><td>Total Evaluated Subjects</td><td class="val">{num_subjects}</td></tr>
                    <tr><td>Total Credit Units</td><td class="val">{total_units:.1f}</td></tr>
                </table>

                <div class="gwa-banner">
                    <span>General Weighted Average (GWA)</span>
                    <span class="gwa-value">
                        {gwa:.2f} &nbsp;·&nbsp;
                        <span style="font-size:14px;font-weight:600;
                                     background:{remark_color};padding:2px 10px;border-radius:10px">
                            {remark}
                        </span>
                    </span>
                </div>

                <div class="rank-bar">
                    <span>Class Rank</span>
                    <span style="font-weight:700;color:#005F30">#{rank} of {len(ranked)}</span>
                </div>

                <button class="print-btn no-print" onclick="window.print()">Print to PDF</button>
            </div>
            """
            display(HTML(report_card_html))

    student_selector.observe(show_student_report, names='value')

    tab = widgets.Tab()
    tab.children = [
        widgets.VBox([student_selector, report_output]),
        scoreboard_output
    ]
    tab.set_title(0, '📋 Student Report')
    tab.set_title(1, '🏆 Rankings')

    display(tab)

    if list(students_data.keys()):
        student_selector.value = list(students_data.keys())[0]
else:
    print("No file was selected.")

Please upload the combined multi-student spreadsheet:


Saving 11 Studetns Report.xlsx to 11 Studetns Report.xlsx
